In [1]:
import pandas as pd
import numpy as np

HOLDOUT_PATH = '2단계_2025홀드아웃_인허가포함.csv.gz'   # 지난번 업로드하신 그 파일
STAGE1_2025_PATH = '2025_frozen_test_risk_score_table.csv'
OUTPUT_PATH = '2단계_2025홀드아웃_1단계포함.csv.gz'


def to_int_quarter(q_str):
    """'2025Q1' -> 20251 (1단계 포맷에 맞춤, 원래 결합 코드와 동일)"""
    year, q = q_str.split('Q')
    return int(year) * 10 + int(q)


# ===== 1. 로드 =====
holdout = pd.read_csv(HOLDOUT_PATH, dtype={'상가업소번호': str, '기준_년분기': str})
print('홀드아웃 로드:', holdout.shape)

stage1_2025 = pd.read_csv(
    STAGE1_2025_PATH,
    usecols=['상권_코드', '서비스_업종_코드', '예측_대상_분기', '예측_폐업위험점수']
)
stage1_2025 = stage1_2025[stage1_2025['예측_대상_분기'] <= 20253]   # 2025Q3까지만
print('1단계 2025 위험점수 테이블(Q3까지):', stage1_2025.shape)

# ===== 2. 기존 전부 NaN인 1단계_위험점수 컬럼 제거 후 새로 조인 =====
holdout['기준_년분기_int'] = holdout['기준_년분기'].apply(to_int_quarter)
holdout = holdout.drop(columns=['1단계_위험점수'])

holdout = holdout.merge(
    stage1_2025,
    left_on=['상권_코드', '서비스_업종_코드', '기준_년분기_int'],
    right_on=['상권_코드', '서비스_업종_코드', '예측_대상_분기'],
    how='left'
)
holdout = holdout.rename(columns={'예측_폐업위험점수': '1단계_위험점수'})

n_total = len(holdout)
n_score_fail = holdout['1단계_위험점수'].isna().sum()
print(f'[1단계 위험점수 결합] 매칭실패 {n_score_fail:,}건 / 전체 {n_total:,}건 '
      f'({n_score_fail/n_total*100:.2f}%)')

# ===== 3. 대체수준 재계산 (기존 로직과 동일: 0=정확매칭/1=상권+분기median/2=분기median) =====
holdout['1단계_위험점수대체수준'] = 0
holdout.loc[holdout['1단계_위험점수'].isna(), '1단계_위험점수대체수준'] = pd.NA

district_median = holdout.groupby(['상권_코드', '기준_년분기'])['1단계_위험점수'].transform('median')
tier1_mask = holdout['1단계_위험점수'].isna() & district_median.notna()
holdout.loc[tier1_mask, '1단계_위험점수대체수준'] = 1
holdout['1단계_위험점수'] = holdout['1단계_위험점수'].fillna(district_median)
n_after_district = holdout['1단계_위험점수'].isna().sum()

quarter_median = holdout.groupby('기준_년분기')['1단계_위험점수'].transform('median')
tier2_mask = holdout['1단계_위험점수'].isna() & quarter_median.notna()
holdout.loc[tier2_mask, '1단계_위험점수대체수준'] = 2
holdout['1단계_위험점수'] = holdout['1단계_위험점수'].fillna(quarter_median)
n_final_na = holdout['1단계_위험점수'].isna().sum()

print(f'[채움] 상권+분기median {n_score_fail-n_after_district:,}건 '
      f'-> 분기median {n_after_district-n_final_na:,}건 '
      f'-> 최종잔여NaN {n_final_na:,}건 (있다면 상권_코드/서비스_업종_코드 자체가 없는 행)')

holdout = holdout.drop(columns=['예측_대상_분기', '기준_년분기_int'])

# ===== 4. 저장 =====
holdout.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', compression='gzip')
print('저장 완료:', OUTPUT_PATH)

홀드아웃 로드: (1612891, 96)
1단계 2025 위험점수 테이블(Q3까지): (227226, 4)
[1단계 위험점수 결합] 매칭실패 849,117건 / 전체 1,612,891건 (52.65%)
[채움] 상권+분기median 235,500건 -> 분기median 73,101건 -> 최종잔여NaN 540,516건 (있다면 상권_코드/서비스_업종_코드 자체가 없는 행)
저장 완료: 2단계_2025홀드아웃_1단계포함.csv.gz
